# Comparación del archivo completo de municipios

`datos/Municipio con ICEE.csv` reemplaza el universo de `datos/Municipio con ICEE menor a 95 por ciento.csv`. Parte de `datos/result.csv` y ya no deja afuera a los municipios con ICEE de 95 % o más.

La llave de todos los cruces es el código DIVIPOLA, con cinco dígitos. Este cuaderno compara el archivo nuevo contra el de referencia y revisa que las columnas añadidas salgan de las fuentes pedidas.

Dos diferencias son esperadas:

- El ICEE del archivo de referencia está redondeado a dos decimales. El nuevo conserva el valor de `result.csv`.
- El valor agregado per cápita del archivo nuevo es `valor agregado / POBLACION`. En el archivo de referencia ese cociente está multiplicado por mil.


In [2]:
from pathlib import Path

import pandas as pd

DATOS = Path("datos")
NUEVO = DATOS / "Municipio con ICEE.csv"
REFERENCIA = DATOS / "Municipio con ICEE menor a 95 por ciento.csv"


def divipola(serie):
    return (
        serie.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.replace("nan", "", regex=False)
        .str.zfill(5)
    )


def numero(serie):
    texto = serie.astype(str).str.strip()
    texto = texto.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    texto = texto.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})
    return pd.to_numeric(texto, errors="coerce")


nuevo = pd.read_csv(NUEVO, sep=";", dtype=str).fillna("")
referencia = pd.read_csv(REFERENCIA, sep=";", dtype=str).fillna("")
nuevo["div"] = divipola(nuevo["DIVIPOLA MUNICIPIO"])
referencia["div"] = divipola(referencia["DIVIPOLA MUNICIPIO"])
print(f"Nuevo: {len(nuevo):,} filas. Referencia: {len(referencia):,} filas.")


Nuevo: 1,122 filas. Referencia: 744 filas.


## Estructura y cobertura

In [3]:
columnas_referencia = [columna for columna in referencia.columns if columna != "div"]
columnas_nuevo = [columna for columna in nuevo.columns if columna != "div"]
assert columnas_nuevo == columnas_referencia
assert nuevo["div"].is_unique
assert referencia["div"].is_unique
assert nuevo["div"].str.fullmatch(r"\d{5}").all()

cruce = referencia.merge(nuevo, on="div", how="outer", indicator=True, suffixes=("_ref", "_nuevo"))
comunes = cruce[cruce["_merge"] == "both"].copy()
solo_nuevo = cruce[cruce["_merge"] == "right_only"].copy()
solo_referencia = cruce[cruce["_merge"] == "left_only"].copy()

assert solo_referencia.empty, "Hay municipios de la referencia que no están en el archivo nuevo"
assert len(comunes) == len(referencia)
assert len(solo_nuevo) == len(nuevo) - len(referencia)
assert (numero(solo_nuevo["ICEE_nuevo"]) >= 95).all()
assert (numero(comunes["ICEE_ref"]) < 95).all()

print(f"Municipios en ambos archivos: {len(comunes):,}")
print(f"Municipios solo en el archivo nuevo, todos con ICEE >= 95: {len(solo_nuevo):,}")
print(f"Municipios solo en la referencia: {len(solo_referencia):,}")


Municipios en ambos archivos: 744
Municipios solo en el archivo nuevo, todos con ICEE >= 95: 378
Municipios solo en la referencia: 0


## Valores de los municipios que ya estaban

In [4]:
tolerancia = {
    "ICEE": 0.005 + 1e-9,
    "valor agregado": 1e-5,
    "Actividades secundarias": 1e-5,
    "Actividades terciarias": 1e-5,
}
exactas = [
    "HOGARES SIN SERVICIO",
    "NBI",
    "CATEGORIA",
    "PDET",
    "ZOMAC",
    "VSS Urbano",
    "VSS Rural",
    "VSS Total",
    "mdm",
    "POBLACION",
    "ANM",
]
resumen = []

for columna in exactas:
    izquierda = comunes[f"{columna}_ref"].str.strip()
    derecha = comunes[f"{columna}_nuevo"].str.strip()
    if columna == "CATEGORIA":
        distintas = izquierda != derecha
        maxima = 0
    else:
        diferencia = (numero(izquierda) - numero(derecha)).abs()
        distintas = ~(
            (numero(izquierda).isna() & numero(derecha).isna())
            | (diferencia.fillna(0) < 1e-9)
        )
        maxima = 0 if diferencia.dropna().empty else diferencia.max()
    resumen.append({"columna": columna, "distintas": int(distintas.sum()), "diferencia máxima": maxima})
    assert not distintas.any(), columna

for columna, tope in tolerancia.items():
    diferencia = (numero(comunes[f"{columna}_ref"]) - numero(comunes[f"{columna}_nuevo"])).abs()
    resumen.append({
        "columna": columna,
        "distintas": int((diferencia.fillna(0) > tope).sum()),
        "diferencia máxima": diferencia.max(),
    })
    assert diferencia.dropna().max() <= tope, columna

nombres = comunes[
    (comunes["NOMBRE DEPARTAMENTO_ref"] != comunes["NOMBRE DEPARTAMENTO_nuevo"])
    | (comunes["NOMBRE MUNICIPIO_ref"] != comunes["NOMBRE MUNICIPIO_nuevo"])
]
assert list(nombres["div"]) == ["11001"]
assert nombres["NOMBRE MUNICIPIO_nuevo"].iloc[0] == "Bogotá, D.C."

assert (numero(comunes["ICEE_nuevo"]).round(2) == numero(comunes["ICEE_ref"])).all()

per_capita_ref = numero(comunes["Valor agregado per capita_ref"])
per_capita_nuevo = numero(comunes["Valor agregado per capita_nuevo"])
escala = (per_capita_ref - per_capita_nuevo * 1000).abs()
resumen.append({
    "columna": "Valor agregado per capita x 1000",
    "distintas": int((escala.fillna(0) > 1e-4).sum()),
    "diferencia máxima": escala.max(),
})
assert escala.dropna().max() <= 1e-4

pd.DataFrame(resumen)


,columna,distintas,diferencia máxima
0,HOGARES SIN SERVICIO,0,0.000000e+00
1,NBI,0,0.000000e+00
2,CATEGORIA,0,0.000000e+00
3,PDET,0,0.000000e+00
4,ZOMAC,0,0.000000e+00
5,VSS Urbano,0,0.000000e+00
6,VSS Rural,0,0.000000e+00
7,VSS Total,0,0.000000e+00
8,mdm,0,0.000000e+00
9,POBLACION,0,0.000000e+00


## Reglas de las columnas nuevas

`ZOMAC` y `VSS Total` no estaban en la lista, pero sí están en el archivo de referencia. `ZOMAC` vale 1 cuando el DIVIPOLA aparece en `municipios_ZOMAC_Colombia_DIVIPOLA.xlsx`. `VSS Total` sale de la columna P de la hoja `ICEE_MPIO`. El MDM se toma del año 2024: la hoja `Base` trae varios años y la columna AS de 2024 es la que coincide con la referencia.


In [5]:
def codigos(serie):
    return set(divipola(serie).dropna())


categoria = pd.read_excel(DATOS / "CATEGORIA_MUNICIPIOS.xlsx", sheet_name="HISTORIA-CAT-MUNIC")
categoria["div"] = divipola(categoria["DIVIPOLA"])
categoria["CATEGORIA"] = categoria["CATEGORIA"].map(lambda valor: "" if pd.isna(valor) else str(valor).strip())
categoria = categoria[["div", "CATEGORIA"]].drop_duplicates("div")

pdet = codigos(pd.read_excel(DATOS / "MunicipiosPDET.xlsx", sheet_name="MunicipiosPDET")["Código DANE Municipio"])
zomac = codigos(
    pd.read_excel(DATOS / "municipios_ZOMAC_Colombia_DIVIPOLA.xlsx", sheet_name="Municipios ZOMAC")[
        "Código DIVIPOLA municipio"
    ]
)

anexo = pd.read_excel(DATOS / "Anexo_ICEE_2024_dpto_mpio.xlsx", sheet_name="ICEE_MPIO")
anexo["div"] = divipola(anexo["Cod. DANE"])
anexo = anexo[["div", "VSS Urbano", "VSS Rural", "VSS Total"]].drop_duplicates("div")

base_mdm = pd.read_excel(DATOS / "Base y diccionario 2024.xlsx", sheet_name="Base")
base_mdm = base_mdm.loc[base_mdm["año"] == 2024, ["cmpio", "mdm"]].copy()
base_mdm["div"] = divipola(base_mdm["cmpio"])
base_mdm = base_mdm[["div", "mdm"]].drop_duplicates("div")

poblacion = pd.read_excel(DATOS / "PPED-AreaMun-2018-2042_VP.xlsx", header=1)
poblacion["div"] = divipola(poblacion["MPIO"])
poblacion = poblacion[["div", "TOTAL"]].drop_duplicates("div")

valor = pd.read_excel(DATOS / "anex-PIBDep-ValorAgreMuni-2011-2024p.xlsx")
valor.columns = [str(columna).replace("\n", "").strip() for columna in valor.columns]
valor["div"] = divipola(valor["Código Municipio"])
valor = valor.rename(
    columns={
        "Actividades secundarias": "Actividades secundarias_fuente",
        "Actividades terciarias": "Actividades terciarias_fuente",
        "Valor agregado": "Valor agregado_fuente",
    }
)
valor = valor[
    [
        "div",
        "Actividades secundarias_fuente",
        "Actividades terciarias_fuente",
        "Valor agregado_fuente",
    ]
].drop_duplicates("div")

armado = nuevo.merge(categoria, on="div", how="left", suffixes=("", "_fuente"))
armado = armado.merge(anexo, on="div", how="left", suffixes=("", "_fuente"))
armado = armado.merge(base_mdm, on="div", how="left", suffixes=("", "_fuente"))
armado = armado.merge(poblacion, on="div", how="left")
armado = armado.merge(valor, on="div", how="left")
assert len(armado) == len(nuevo)

assert (armado["CATEGORIA"].fillna("") == armado["CATEGORIA_fuente"].fillna("")).all()
assert (numero(armado["PDET"]) == armado["div"].isin(pdet).astype(int)).all()
assert (numero(armado["ZOMAC"]) == armado["div"].isin(zomac).astype(int)).all()
for columna in ["VSS Urbano", "VSS Rural", "VSS Total"]:
    diferencia = (numero(armado[columna]) - pd.to_numeric(armado[f"{columna}_fuente"])).abs()
    assert diferencia.max() < 1e-6, columna
diferencia_mdm = numero(armado["mdm"]) - pd.to_numeric(armado["mdm_fuente"])
assert (diferencia_mdm.dropna().abs().max() < 1e-6) and (
    numero(armado["mdm"]).isna() == pd.to_numeric(armado["mdm_fuente"]).isna()
).all()
assert (numero(armado["POBLACION"]) == pd.to_numeric(armado["TOTAL"])).all()
for columna in ["Actividades secundarias", "Actividades terciarias", "Valor agregado"]:
    destino = "valor agregado" if columna == "Valor agregado" else columna
    diferencia = (numero(armado[destino]) - pd.to_numeric(armado[f"{columna}_fuente"])).abs()
    assert diferencia.max() < 1e-6, columna

cociente = numero(armado["valor agregado"]) / numero(armado["POBLACION"])
diferencia_pc = (numero(armado["Valor agregado per capita"]) - cociente).abs()
assert diferencia_pc.max() < 1e-6

anm_referencia = dict(zip(referencia["div"], numero(referencia["ANM"])))
anm_esperado = nuevo["div"].map(anm_referencia).fillna(0)
assert (numero(nuevo["ANM"]) == anm_esperado).all()
assert (numero(solo_nuevo["ANM_nuevo"]) == 0).all()

print("Categoría, PDET, ZOMAC, VSS, MDM 2024, población, valor agregado y ANM coinciden con sus fuentes.")
print("El valor agregado per cápita es el valor agregado dividido por la población.")


Categoría, PDET, ZOMAC, VSS, MDM 2024, población, valor agregado y ANM coinciden con sus fuentes.
El valor agregado per cápita es el valor agregado dividido por la población.


Si las celdas anteriores terminan sin error, el archivo nuevo conserva los 744 municipios de la referencia y agrega el resto del universo de `result.csv`.